In [4]:
from http.server import BaseHTTPRequestHandler, HTTPServer
import os
import json

# library we use to easily extract query parameters
from urllib.parse import urlparse, parse_qs


#You should not change this lines.
#They define the parameters required to run the server.
hostName = "localhost"
serverPort = 8080

#we load all the data now once so it is not needed to load it in each request
with open("json_rand_users_5000.json", encoding="utf-8") as f:
       users = json.load(f)["results"]

#we also load once the valid user IDs and access tokens (needed in milestone 7)
#we save them in a dictionary like {"9123": "$P$BZlw7PF1j.XDezr9sUj6moCjBlSx/e0", ...}
access_tokens = {}
with open("file_access_token_M7.csv", encoding="utf-8") as f:
       next(f) #skip the first line (header: User_ID;Access_Token)
       for line in f:
              user_id, token = line.strip().split(";")
              access_tokens[user_id] = token


#MILESTONE 1
#endpoint => http://localhost:8080/test_connectivity_M1

#You just need to test the connectivity that returns the following JSON
#{"results": [{"Connectivity": "OK"}]}

#JSON TO DISPLAY IN CLIENT:
#{"results": [{"Connectivity": "OK"}]}

def test_connectivity_M1(text):
    results = {}
    json_result={}
    array = []             
    json_result['Connectivity']="OK"
    array.append(json_result)
    results['results'] = array
    api_answer=json.dumps(results, ensure_ascii=False)
    output_code=200
    return api_answer,output_code

#MILESTONE 2
#endpoint => http://localhost:8080/count_countries_M2
#Country Count. Provide the number of entries per country 
#in the dataset in a sorted way. First the most frequent country.
#TEXT TO DISPLAY IN CLIENT:
# 1: TR(317)
# 2: DE(317)
# 3: NO(315)
# 4: BR(312)
# 5: IR(311)
# 6: CA(297)
# 7: FR(297)
# 8: CH(296)
# 9: IE(293)
# 10: DK(290)
# 11: NZ(288)
# 12: AU(285)
# 13: ES(281)
# 14: GB(278)
# 15: NL(276)
# 16: FI(276)
# 17: US(271)

def country_count_M2(text):
    results = {}
    array = []
    counts = {}

    for user in users:
        country = user['nat']
        if country in counts:
            counts[country] += 1
        else:
            counts[country] = 1
    
    # Sorting results
    # (we were unsure from the statement if desired to do so in server or client side,
    # so we also include the sorting in the client side as a comment)
    counts = sorted(counts.items(), key=lambda item: item[1], reverse=True)
    
    for country, count in counts:
        array.append({"country": country, "count": count})

    results['results'] = array
    api_answer=json.dumps(results, ensure_ascii=False)
    output_code=200
    return api_answer,output_code
    
#MILESTONE 3    
#endpoint=>http://localhost:8080/count_countries_M3?countries=DK,ES,GB,IE,IR,NO,NL,NZ

#It is the same than before but in case it just returns the result for the provided countries.
#The list of available countries in the raduser API (field nat) is:
#AU, BR, CA, CH, DE, DK, ES, FI, FR, GB, IE, IR, NO, NL, NZ, TR, US
#If not countries parameter is provided you have to provide the results for all the countries
#NOTE: NO NEED TO IMPLEMENT CONTROL OF ERRORS

#TEXT TO DISPLAY IN CLIENT THAT ONLY CONSIDERS THE COUNTRIES IN THE QUERY:
# 1: NO(315)
# 2: IR(311)
# 3: IE(293)
# 4: DK(290)
# 5: NZ(288)
# 6: ES(281)
# 7: GB(278)
# 8: NL(276)

def country_count_M3(text):   
    results = {}
    array = []

    # extract the countries list, we assume input in correct format as per the note above             
    params = parse_qs(urlparse(text).query)
    requested_countries = params['countries'][0].split(',')

    counts = {}
    for user in users:
        country = user['nat']
        # Only consider users from requested_countries
        if country in requested_countries:
            if country in counts:
                counts[country] += 1
            else:
                counts[country] = 1
    
    # Sorting results
    # (we were unsure from the statement if desired to do so in server or client side,
    # so we also include the sorting in the client side as a comment)
    counts = sorted(counts.items(), key=lambda item: item[1], reverse=True)
    
    for country, count in counts:
        array.append({"country": country, "count": count})

    results['results'] = array
    api_answer=json.dumps(results, ensure_ascii=False)
    output_code=200
    return api_answer,output_code

#MILESTONE 4    
#endpoint=>http://localhost:8080/age_groups_M4 
#You need to deliver the number of users in the data set
#within the following age groups: <18, 18-29, 30-49, 50-64, >64.

#ANSWER IN CLIENT
# Age <18: 0
# Age 18-29: 634
# Age 30-49: 1762
# Age 50-64: 1452
# Age >64: 1152

def age_groups_M4(text):
    results = {}
    array = []

    groups = {"<18": 0, "18-29": 0, "30-49": 0, "50-64": 0, ">64": 0}
    for user in users:
        age = user['dob']['age']
        if 0 <= age < 18: #<=0 to remove erroneous data (negative ages)
            groups["<18"] += 1
        elif age <= 29:
            groups["18-29"] += 1
        elif age <= 49:
            groups["30-49"] += 1
        elif age <= 64:
            groups["50-64"] += 1
        else:
            groups[">64"] += 1

    for group, count in groups.items():
        array.append({"group": group, "count": count})
        
    results['results'] = array
    api_answer=json.dumps(results, ensure_ascii=False)
    output_code=200
    return api_answer,output_code

#MILESTONE 5
#endpoint => http://localhost:8080/users_M5?countries=FR,GB,IE,IR,NO,NL,NZ,TR,US&gender=female&month=01,02,03,04,05&age=30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45
#You need to find users meeting the list of parameters in the query.
#The query should work with 1, 2, 3 or 4 of the fields combined in whatever

#countries=AU,BR,CA,CH,DE,DK,ES,FI,FR,GB,IE,IR,NO,NL,NZ,TR,US  [use the nat field for this] (if not defined consider all countries)
#gender=male,female (if not used consider all genders)
#month= 01,02,03,04,05,06,07,08,09,10,11,12 (if not used consider all the months)
#age:Any value between 18 and 99 (if not used consider all ages)

#NOTE1: IT IS NOT NEEDED TO DO CONTROL ERRORS. 
#NOTE2: You need to provide the total number of users and the 100 first users in the JOSN as follows:
#NOTE3: some times the query selection my obtain less than 100 users then print all of them

#TEXT TO DISPLAY IN CLIENT:

# NUM. USERS:165
# User 1:
# 	-Name: Linne Hoogkamer
# 	-Gender: female
# 	-Age: 40
# 	-Country: Netherlands
# 	-Postcode: 42173
# 	-Email: linne.hoogkamer@example.com
# User 2:
# 	-Name: Enola Martin
# 	-Gender: female
# 	-Age: 43
# 	-Country: France
# 	-Postcode: 93894
# 	-Email: enola.martin@example.com
# User 3:
# 	-Name: Mathilda Skjevik
# 	-Gender: female
# 	-Age: 42
# 	-Country: Norway
# 	-Postcode: 6630
# 	-Email: mathilda.skjevik@example.com
# User 4:
# 	-Name: Valentine Thomas
# 	-Gender: female
# 	-Age: 35
# 	-Country: France
# 	-Postcode: 84452
# 	-Email: valentine.thomas@example.com
#.....
#.....
#.....
#User 100:
# 	-Name: Cecilia Asphaug
# 	-Gender: female
# 	-Age: 42
# 	-Country: Norway
# 	-Postcode: 5052
# 	-Email: cecilia.asphaug@example.com

def users_M5(text):
    results = {}
    array = []
    
    # Parse provided parameters
    params = parse_qs(urlparse(text).query)

    # We initialize to all possible values according to the statement
    """This way even if a parameter is not included in the request 
    we ensure only to include in the response users with all values inside the possible values,
    which we understood is the desired behaviour (i.e. we will NOT include a person from country "ZZ")"""

    valid_countries = ["AU", "BR", "CA", "CH", "DE", "DK", "ES", "FI", "FR", "GB", "IE", "IR", "NO", "NL", "NZ", "TR", "US"]
    valid_genders = ["male", "female"]
    valid_months = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]
    valid_ages = [str(a) for a in range(18, 100)]

    # Then for all provided in the request we overwrite the list
    if 'countries' in params:
        valid_countries = params['countries'][0].split(',')
    if 'gender' in params:
        valid_genders = params['gender'][0].split(',')
    if 'month' in params:
        valid_months = params['month'][0].split(',')
    if 'age' in params:
        valid_ages = params['age'][0].split(',')

    # We filter users in the same loop than write them to the results array
    # as we wanted to solve the query in a single data pass for efficiency
    num_users = 0
    for user in users:
        if (user['nat'] in valid_countries and user['gender'] in valid_genders 
            and user['dob']['date'][5:7] in valid_months and str(user['dob']['age']) in valid_ages):
            
            num_users += 1
            # Only the first 100 matching users are included (if fewer match, all of them are)
            if num_users <= 100:
                array.append({"name": user['name']['first'] + " " + user['name']['last'],
                            "gender": user['gender'],
                            "age": user['dob']['age'],
                            "country": user['location']['country'],
                            "postcode": user['location']['postcode'],
                            "email": user['email']})

    results['num_users'] = num_users
    results['results'] = array
    api_answer=json.dumps(results, ensure_ascii=False)
    output_code=200
    return api_answer,output_code

#MILESTONE 6
#We have to repeat the same exercise than in M5 but this time we need to implement control error
#FORMAT CONTROL:
# Countries: If a country is not in the available list. Generate an error and inform the user which country(s) is(are) wrong with a mesasge
# Gender: If a gender is not in the list of genders. Generate an error and inform the user which gender(s) is(are) wrong with a message
# Month: If the month is not a valid value (e.g., 15). Generate an error and inform the user which month(s) is(are) wrong with a message
# Age: if the age is not in the range 18-99 it is a wrong value. Generate an error and inform the user which age(s) is(are) wrong with a message
# Field: If after ? there is an field (e.g., extra_field) different that countries, gender, age and month report the error.
# The output message should be json file with an error per category including all the wrong values employed.
#
#NOTES:
#1- The succeed of the process should report a status code 200.
#2- The failure of the process due to an error should report the status 400 (Bad request)
#3- You need to include all the errors so you should not stop the verification of the format when you find the first error.

#TEXT TO DISPLAY IN CLIENT (it is assuming all errors are present in the created json based on the following request: )
 
#JOSN output
#{"results": [{"Country Error": "The following country IDs are not supported: IT,SW,MX", 
#"Gender Error": "The following gender options are not supported: women,men", 
#"Month Error": "The following month options are not supported: 15,23", 
#"Age Error": "The following month options are not supported: 12,15,123", 
#"Field Error": "The following fields are not valid: city"}]}

#IF THERE IS AN ERROR ONLY IN 1 or 2 FIELDS THE JSON SHOULD ONLY INCLUDE THOSE ERRORS
def users_control_error_M6(text):
    results = {}
    array = []
    errors = {}

    params = parse_qs(urlparse(text).query)

    # we are going to use sets to efficiently identify if any of the provided values is out of the expected ones
    valid_fields = {"countries", "gender", "month", "age"}
    valid_countries = {"AU", "BR", "CA", "CH", "DE", "DK", "ES", "FI", "FR", "GB", "IE", "IR", "NO", "NL", "NZ", "TR", "US"}
    valid_genders = {"male", "female"}
    valid_months = {"01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"}
    valid_ages = {str(a) for a in range(18, 100)}

    # We check all possible errors one by one, independently if there was already a previous error
    if 'countries' in params:
        wrong = [c for c in params['countries'][0].split(',') if c not in valid_countries]
        if wrong:
            errors["Country Error"] = "The following country IDs are not supported: " + ",".join(wrong)

    if 'gender' in params:
        wrong = [g for g in params['gender'][0].split(',') if g not in valid_genders]
        if wrong:
            errors["Gender Error"] = "The following gender options are not supported: " + ",".join(wrong)

    if 'month' in params:
        wrong = [m for m in params['month'][0].split(',') if m not in valid_months]
        if wrong:
            errors["Month Error"] = "The following month options are not supported: " + ",".join(wrong)

    if 'age' in params:
        wrong = [a for a in params['age'][0].split(',') if a not in valid_ages]
        if wrong:
            errors["Age Error"] = "The following age options are not supported: " + ",".join(wrong)
    
    wrong_fields = [f for f in params if f not in valid_fields]
    if wrong_fields:
        errors["Field Error"] = "The following fields are not valid: " + ",".join(wrong_fields)

    if errors:
        array.append(errors)
        results['results'] = array
        api_answer = json.dumps(results, ensure_ascii=False)
        output_code = 400
        return api_answer, output_code

    # If there were no errors detected, valid request, call milestone 5
    return users_M5(text)

#MILESTONE 7
#endpoint => http://localhost:8080/access_token_M7?access_token=$P$BZlw7PF1j.XDezr9sUj6moCjBlSx/e0&countries=FR,GB,IE,IR,NO,NL,NZ,TR,US
#&user_ID=9123&gender=male,female&month=01,05,06,12&age=30,31,32,33,34,35

#We have to repeat the same exercise than in M5 but this time we need to controll whether the userID and access token are correct.
#To this end we will use a query as follow where the access token parameter appears as another field. But it is a compulsory field.
#So we basically need to verify if the access token is there and if so we call the M6 endpoint.
#The URL will look like as follow:
#http://localhost:8080/access_token_M7?userID=127821&access_token=XXASJasnsna?.12123jk2amsass?&countries=FR,ES&age=18,19,20,21,22&gender=male
#You need to validate the user ID and access token in the list of valids access tokens.


#TEXT OTPIONS TO DISPLAY IN CLIENT FOR ERRORS (if access token and user ID correct then deliver answer associated to M6)

#1- No fields included (could be one of them or both of them) 
#{"results": [{"user_ID Error": "user_ID field not included in the parameters.", 
#"Access_Token Error": "Access_Token field not included in the parameters."}]}

#2- UserID does not exist
#{"results": [{"user_ID Error": "user_ID 2131 not found in the database."}]}

#3- User ID exists but access_token wrong
#{"results": [{"access_token Error": "access_token 7423hjmn23nsd.121 wrong for the userID 9123"}]}

def access_token_M7(text):
    results = {}
    array = []
    errors = {}

    params = parse_qs(urlparse(text).query)

    # 1- We check that both compulsory fields are included (if both are missing we report both)
    if 'user_ID' not in params:
        errors["user_ID Error"] = "user_ID field not included in the parameters."
    if 'access_token' not in params:
        errors["Access_Token Error"] = "Access_Token field not included in the parameters."

    # We can only check the values if both fields are included
    if not errors:
        user_id = params['user_ID'][0]
        token = params['access_token'][0]
        # 2- The user ID has to exist in the file of valid access tokens
        if user_id not in access_tokens:
            errors["user_ID Error"] = "user_ID " + user_id + " not found in the database."
        # 3- The access token has to be the one saved for that user ID
        elif access_tokens[user_id] != token:
            errors["access_token Error"] = "access_token " + token + " wrong for the userID " + user_id

    # Same as in milestone 6: if there is any error we return it with status 400
    if errors:
        array.append(errors)
        results['results'] = array
        api_answer = json.dumps(results, ensure_ascii=False)
        output_code = 400
        return api_answer, output_code

    # If user ID and access token are correct, we remove them from the query and call milestone 6
    # (if not removed, milestone 6 would report them as a "Field Error")
    del params['user_ID']
    del params['access_token']
    new_text = "/users_control_error_M6?" + "&".join(field + "=" + values[0] for field, values in params.items())
    return users_control_error_M6(new_text)


class MyServer(BaseHTTPRequestHandler):
    def do_GET(self):
        end_point=self.path
        
        if(end_point.startswith("/test_connectivity_M1")):
            api_answer,output_code= test_connectivity_M1(end_point)
            
        elif(end_point.startswith("/count_countries_M2")):
            api_answer,output_code= country_count_M2(end_point)

        
        elif(end_point.startswith("/count_countries_M3")):
            api_answer,output_code= country_count_M3(end_point)

            
        elif(end_point.startswith("/age_groups_M4")):
            api_answer,output_code= age_groups_M4(end_point)

            
        elif(end_point.startswith("/users_M5")):
            api_answer,output_code= users_M5(end_point)

        elif(end_point.startswith("/users_control_error_M6")):
            api_answer,output_code= users_control_error_M6(end_point)
        
        elif(end_point.startswith("/access_token_M7")):
            api_answer,output_code= access_token_M7(end_point)

        
        else:
            results = {}
            json_error={}
            array = []             
            json_error['Endpoint Error']="The provided endpoint does not exist."
            array.append(json_error)
            results['results'] = array
            api_answer=json.dumps(results, ensure_ascii=False)
            output_code=400
        
        self.send_response(output_code)
        self.send_header("Content-type", "text/json")
        self.end_headers() 
        self.wfile.write(bytes(api_answer, "utf-8"))
        
if __name__ == "__main__":        
    webServer = HTTPServer((hostName, serverPort), MyServer)
    print("Server started http://%s:%s" % (hostName, serverPort))

    try:
        webServer.serve_forever()
    except KeyboardInterrupt:
        pass

    webServer.server_close()
    print("Server stopped.")



Server started http://localhost:8080


127.0.0.1 - - [30/Sep/2026 20:59:53] "GET /test_connectivity_M1 HTTP/1.1" 200 -
127.0.0.1 - - [30/Sep/2026 20:59:53] "GET /count_countries_M2 HTTP/1.1" 200 -
127.0.0.1 - - [30/Sep/2026 20:59:53] "GET /count_countries_M3?countries=AU,BR,CA HTTP/1.1" 200 -
127.0.0.1 - - [30/Sep/2026 20:59:53] "GET /age_groups_M4 HTTP/1.1" 200 -
127.0.0.1 - - [30/Sep/2026 20:59:53] "GET /users_M5?countries=FR,GB,IE,IR,NO,NL,NZ,TR,US&gender=female&month=01,02,03,04,05&age=30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45 HTTP/1.1" 200 -
127.0.0.1 - - [30/Sep/2026 20:59:53] "GET /users_control_error_M6?countries=FR,GB,IE,IR,NO,NL,NZ,TR,US,IT,SW,MX&gender=women,men&month=01,05,06,12,15,23&age=30,31,32,33,34,35,36,37,38,39,40,12,15,123&city=Madrid HTTP/1.1" 400 -
127.0.0.1 - - [30/Sep/2026 20:59:53] "GET /users_control_error_M6?countries=FR,GB,IE,IR,NO,NL,NZ,TR,US&gender=female&month=01,02,03,04,05&age=30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45 HTTP/1.1" 200 -
127.0.0.1 - - [30/Sep/2026 20:59:53] "GET /acc

Server stopped.
